# Nested JSON Insurance Claims with PySpark

Explore nested JSON processing with Spark: flatten nested treatment arrays, explode treatment and medicine records, aggregate costs at claim level, and calculate the final claim amount.

**Formula:** treatment cost + medicine cost - deductions.

Expected input: `data/insurance_claim.json` relative to this project folder.

## 1. Create Spark session and read JSON

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("InsuranceClaimsAnalysis").master("local[*]").getOrCreate())

df = spark.read.json("data/insurance_claim.json")  # JSON Lines: one claim record per line
df.printSchema()
df.show(truncate=False)

## 2. Flatten and explode nested treatments and medicines

The medicine explode repeats treatment columns once per medicine. Keep that in mind when aggregating treatment cost.

In [ ]:
step1_df = (
    df
    .withColumn("flattened_treatments", F.flatten(F.col("treatments")))
    .withColumn("exploded_treatments", F.explode(F.col("flattened_treatments")))
    .select(
        "claim_id",
        F.col("exploded_treatments.treatment_id").alias("treatment_id"),
        F.col("exploded_treatments.treatment_name").alias("treatment_name"),
        F.col("exploded_treatments.cost").alias("cost"),
        F.col("exploded_treatments.units").alias("units"),
        F.explode(F.col("exploded_treatments.medicines")).alias("medicine_exploded")
    )
)
step1_df.show(truncate=False)

## 3. Aggregate treatment cost

Select distinct treatment-level records before aggregation so medicine rows do not multiply treatment amounts. In production, validate the business key and duplicate semantics.

In [ ]:
treatment_df = step1_df.select("claim_id", "treatment_id", "treatment_name", "cost", "units").distinct()
total_treatment_cost_final_df = (
    treatment_df.withColumn("treatment_total", F.col("cost") * F.col("units"))
    .groupBy("claim_id").agg(F.sum("treatment_total").alias("total_treatment_cost"))
)
total_treatment_cost_final_df.show(truncate=False)

## 4. Aggregate medicine cost

In [ ]:
medicine_df = step1_df.select(
    "claim_id", "treatment_id",
    F.col("medicine_exploded.medicine_name").alias("medicine_name"),
    F.col("medicine_exploded.price").alias("medicine_price"),
    F.col("medicine_exploded.quantity").alias("medicine_quantity")
)
Total_medicine_cost_claim_level = medicine_df.groupBy("claim_id").agg(
    F.sum(F.col("medicine_price") * F.col("medicine_quantity")).alias("total_medicine_cost")
)
Total_medicine_cost_claim_level.show(truncate=False)

## 5. Aggregate deductions

In [ ]:
total_deductions_df = (
    df.select("claim_id", F.explode(F.col("deductions")).alias("deduction_exploded"))
    .withColumn("deduction_amount", F.col("deduction_exploded.amount").cast("double"))
)
deductions_final_df = total_deductions_df.groupBy("claim_id").agg(
    F.sum("deduction_amount").alias("total_deductions")
)
deductions_final_df.show(truncate=False)

## 6. Join claim-level totals and calculate final amount

Aggregate each component at claim grain before joining to avoid multiplying treatment, medicine, and deduction rows. This example uses inner joins; choose join behavior based on how missing components should be handled in your use case.

In [ ]:
final_cost = (
    total_treatment_cost_final_df
    .join(Total_medicine_cost_claim_level, ["claim_id"], "inner")
    .join(deductions_final_df, ["claim_id"], "inner")
)
final_claim_cost_df = final_cost.withColumn(
    "final_cost",
    F.col("total_treatment_cost") + F.col("total_medicine_cost") - F.col("total_deductions")
)
final_claim_cost_df.show(truncate=False)

## Key takeaways
- `flatten()` removes one level of nested arrays.
- `explode()` turns array elements into rows.
- Track DataFrame grain to avoid double counting.
- Aggregate components independently before joining.
- Decide explicitly how null or missing components should be handled.